#Introducción

En este nootbook nos proponemos realizar el análisis exploratorio de un conjunto de datos correspondiente a una plataforma de streaming. El dataset contiene información de usuarios suscriptos al servicio, incluyendo características demográficas, hábitos de consumo y datos relacionados con el uso de la plataforma.

Entre las variables analizadas se encuentran la edad de los usuarios, el país de residencia, el tipo de plan de suscripción contratado, el género de contenido favorito, la cantidad de minutos de visualización mensual, la cantidad de tickets de soporte generados y la fecha del último inicio de sesión.

El objetivo principal del análisis exploratorio es conocer la estructura del conjunto de datos, identificar posibles problemas de calidad y detectar inconsistencias que puedan afectar futuros análisis estadísticos o modelos predictivos. Para ello, se realizó una inspección de las variables, la búsqueda de valores faltantes, registros duplicados, errores de carga, inconsistencias en las categorías y valores atípicos.

Este proceso permite comprender mejor las características del dataset y evaluar su nivel de calidad antes de llevar a cabo etapas posteriores de análisis, visualización o modelado de datos.

In [ ]:
import pandas as pd

importamos la libreria pandas para poder manipular los datos

In [ ]:
from google.colab import files
upload = files.upload()

Saving streaming_users_dirty.json to streaming_users_dirty.json


cargamos el Dataset en formato tipo Json a traves de codigo e inciamos la exploración general del mismo

In [ ]:
df = pd.read_json(next(iter(upload)))
df.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


inforamcion general de las primeras 5 filas

In [ ]:
df.shape

(8160, 8)

cantidad de filas y columnas

In [ ]:
df = df.rename(columns={
    "user_id": "id_usuario",
    "age": "edad",
    "subscription_plan": "plan_suscripcion",
    "monthly_watch_time_mins": "minutos_visualizacion_mensual",
    "country": "pais",
    "favorite_genre": "genero_favorito",
    "last_login_date": "fecha_ultimo_inicio_sesion",
    "customer_support_tickets": "tickets_soporte_cliente"
})

df.head()

,id_usuario,edad,plan_suscripcion,minutos_visualizacion_mensual,pais,genero_favorito,fecha_ultimo_inicio_sesion,tickets_soporte_cliente
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


tradcucimos al español todas las variables para poder interpretarlas

In [ ]:
df.dtypes

,0
id_usuario,int64
edad,int64
plan_suscripcion,object
minutos_visualizacion_mensual,float64
pais,object
genero_favorito,object
fecha_ultimo_inicio_sesion,object
tickets_soporte_cliente,int64


identificamoes los tipos de variable

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id_usuario                     8160 non-null   int64  
 1   edad                           8160 non-null   int64  
 2   plan_suscripcion               8160 non-null   object 
 3   minutos_visualizacion_mensual  7967 non-null   float64
 4   pais                           8160 non-null   object 
 5   genero_favorito                7920 non-null   object 
 6   fecha_ultimo_inicio_sesion     7840 non-null   object 
 7   tickets_soporte_cliente        8160 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 510.1+ KB


detectamos que en "minutos_visualizacion_mensual, genero_favorito, ticket_soporte_cliente" existen datos faltantes.

In [ ]:
df.duplicated().sum()

np.int64(126)

existen 126 datos duplicados

In [ ]:
df["plan_suscripcion"].value_counts()

,count
plan_suscripcion,
Básico,3450
Estándar,2711
Premium,1519
basico,60
BASICO,52
Basic,52
básico,50
Std,48
Estándar,46


In [ ]:
df["pais"].value_counts()

,count
pais,
Brasil,1132
Chile,1132
México,1129
Uruguay,1124
Perú,1120
Colombia,1116
Argentina,1087
colombia,27
méxico,25


In [ ]:
df["genero_favorito"].value_counts()

,count
genero_favorito,
Comedia,1112
Drama,1105
Documental,1095
Thriller,1090
Romance,1090
Acción,1082
Crime,1067
Action,22
COMEDIA,19


Detectamos errores de tipeo y repiticiones en las columnas categoricas

In [ ]:
df['fecha_ultimo_inicio_sesion'].value_counts()

,count
fecha_ultimo_inicio_sesion,
2026-15-03,20
0000-00-00,17
2029-01-01,15
not_available,14
31-02-2022,13
...,...
2018-11-07,1
2022-03-20,1
2019-01-19,1


se detectaron fechas incongruentes

In [ ]:
fechas = pd.to_datetime(
    df["fecha_ultimo_inicio_sesion"],
    errors="coerce"
)

print("Fechas inválidas:", fechas.isna().sum())
print("Fechas futuras:", (fechas > pd.Timestamp.today()).sum())

Fechas inválidas: 769
Fechas futuras: 15


In [ ]:
df['edad'].unique()

array([ 39,  37,  28,  43,  51,  20,  31,  36,  23,  46,  15,  45,  29,
        24,  35,  32,  30,  17,  33,  18,  40,  42,  22,  44,  41,  59,
        26,  25,  27,  53,   0,  55,  34,  47,  16,  21,  13,  38,  56,
        19,  48,  14,  57,  52,  64,  60,  58, 130,  54,  50,  49,  61,
        66,  73, 150,  -5,  68,   4,  69,  62,  72,  63,  67,  65,  76,
        80,  74,  71,  70])

existen edades imposibles

In [ ]:
df.describe()

,id_usuario,edad,minutos_visualizacion_mensual,tickets_soporte_cliente
count,8160.000000,8160.000000,7967.000000,8160.000000
mean,13995.433824,34.096814,1107.346153,1.800980
std,2310.810660,14.511304,5310.442604,11.334969
min,10000.000000,-5.000000,-120.000000,-1.000000
25%,11987.750000,25.000000,489.200000,0.000000
50%,13998.500000,33.000000,757.400000,1.000000
75%,15997.250000,42.000000,1045.700000,1.000000
max,17999.000000,150.000000,99999.000000,150.000000


In [ ]:
df["id_usuario"].duplicated().sum()

np.int64(160)

In [ ]:
import json
df_json = json.loads(upload['streaming_users_dirty.json'])
df = pd.DataFrame(df_json)
df = df.rename(columns={'user_id': 'id_usuario'})

ids_repetidos = df[df["id_usuario"].duplicated(keep=False)]

ids_repetidos.sort_values("id_usuario")

,id_usuario,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
37,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8133,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8089,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
52,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
59,10059,39,Estándar,2976.6,Brasil,DRAMA,2024-06-22,1
...,...,...,...,...,...,...,...,...
8125,17784,38,Estándar,492.2,México,Documental,2021-10-04,1
8053,17902,30,Básico,505.7,Brasil,Thriller,2025-06-15,0
7902,17902,30,Básico,505.7,Brasil,THRILLER,2025-06-15,0
7994,17994,24,Básico,437.0,México,Romance,2020-08-12,0


detectamos Id_usuarios duplicados

In [ ]:
df[df["id_usuario"]== 17994]

,id_usuario,edad,plan_suscripcion,minutos_visualizacion_mensual,pais,genero_favorito,fecha_ultimo_inicio_sesion,tickets_soporte_cliente
7994,17994,24,Básico,437.0,México,Romance,2020-08-12,0
8046,17994,24,Básico,437.0,México,Romance,2020-08-12,0


#Informe sobre exploración inicial del dataset

Durante la inspección inicial se analizaron las variables, los tipos de datos y la presencia de valores faltantes, duplicados e inconsistencias.

El conjunto de datos contiene variables numéricas, categóricas y temporales relacionadas con usuarios de una plataforma de streaming.


---

#Valores Faltantes

Se detectó la presencia de datos faltantes en varias variables del conjunto de datos.

Variable	Cantidad de valores faltantes

minutos_visualizacion_mensual	193
genero_favorito	240
fecha_ultimo_inicio_sesion	320


La existencia de valores nulos puede afectar cálculos estadísticos, visualizaciones y modelos predictivos.


---

#Registros Duplicados

El análisis permitió detectar:

126 filas completamente duplicadas.

160 identificadores de usuario repetidos.


La presencia de registros duplicados puede provocar una sobre representación de ciertos usuarios y alterar los resultados del análisis.


---

#Inconsistencias en Variables Categóricas

Se observaron diferentes formas de representar una misma categoría.

Plan de suscripción

Se encontraron variantes como:

BASICO

basico

Basic

Premium

Premiun


---



Se detectaron diferentes formas de registrar los países:

ARG

Argentina

BRA

Brazil

MEX

México





---

Se identificaron distintas escrituras para una misma categoría:

Action

accion

Acción

thriler

crime


---

#Problemas en Variables Numéricas

Edad

Se detectaron edades fuera de los rangos esperados, incluyendo valores negativos o excesivamente altos. Esto podria deberse a errores de carga


---

Minutos de visualización

Se encontraron valores negativos o extremadamente altos. Estos outliers pueden considerarse atípicos o erróneos.


---

Tickets de soporte

Se observaron cantidades negativas de tickets, situación que no resulta coherente desde el punto de vista empresarial.


---

#Problemas en Variables de Fecha

El análisis de las fechas permitió identificar:

Fechas con formatos inválidos.

Registros imposibles de convertir a formato fecha.

Fechas ubicadas en el futuro.


Se detectaron aproximadamente 769 fechas inválidas y 15 fechas futuras.


---

#Impacto de los Problemas Detectados

Los errores encontrados pueden generar:

Resultados estadísticos incorrectos.

Sesgos en los análisis.

Duplicación de información.

Errores en visualizaciones.

Problemas en modelos de aprendizaje automático.

Conclusiones erróneas sobre el comportamiento de los usuarios.



---

#Conclusiones

La exploración del conjunto de datos permitió identificar diversos problemas de calidad que afectan la confiabilidad de la información.

Las principales observaciones fueron:

Existencia de valores faltantes en variables importantes.

Presencia de registros duplicados.

Inconsistencias en las variables categóricas.

Valores numéricos fuera de rangos esperados.

Fechas inválidas o inconsistentes.

Falta de estandarización en algunas variables.


En conclusión, el dataset presenta multiples problemas e inconsistencias en sus datos. La detección temprana de estas inconsistencias permite comprender las limitaciones de los datos y evaluar su impacto en futuras etapas del proyecto.